# Importing and preparing flatfox rental apartment data (buildings built before 1990)

**This Jupyter notebook does the following:**

- Imports web-scraped flatfox rental apartment data (buildings built in 1990 or earlier) from a CSV file
- Inspects the number of rows/columns and the data types
- Extracts the number of rooms, living area and rental price from raw strings using regular expressions (regex)
- Flags rents per week, furnished and temporary listings (they are kept, not deleted)
- Counts missing and duplicated values
- Removes apartments built in 1990, because Topic 1 covers buildings built **before** 1990
- Topic 1 extensions: cleans construction/renovation years, computes rent per m², extracts postcode and locality, and assigns renovation groups (not renovated / general renovation / energy renovation) using keywords
- Adds the boolean column `minergie` (flatfox attribute from `01_flatfox_web_api.ipynb` or "Minergie" in the description)
- Saves the prepared data to `Data/flatfox_pre1991_prepared.csv`

The structure follows `Week_08/apartments_data_preparation_winterthur.ipynb`.

## Libraries and settings

In [1]:
# Libraries
import os
import re
import fnmatch
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

c:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Importing data

In [2]:
# Show .csv - files in the folder 'Data'
flist = fnmatch.filter(os.listdir('./Data'), '*.csv')
for i in flist:
    print(i)

# Read the data to a pandas data frame
# (descriptions contain line breaks inside quotes; read_csv handles this)
df = pd.read_csv('./Data/flatfox_switzerland_rent_apartments_pre1991.csv', sep=',', encoding='utf-8')

# Show first records of data frame
df.head()

AMTOVZ_CSV_WGS84.csv
apartments_pre1990_clean.csv
apartments_pre1990_combined.csv
apartments_pre1990_enriched.csv
apartments_pre1990_renovation_details.csv
flatfox_api_attributes.csv
flatfox_pre1991_prepared.csv
flatfox_switzerland_rent_apartments_pre1991.csv
immoscout24_mieten_400.csv
immoscout24_pre1990_prepared.csv


,web_scraper_order,web_scraper_start_url,rooms_card,address,price_card,detail_url,detail_url-href,gross_rent,rooms_detail,living_area,year_built,year_renovated,description
0,1794995628-1,https://flatfox.ch/en/search/?east=10.5&is_swa...,3 rooms apartment6614 Brissago,6614 Brissago,Rental\n1’200 CHF,3 rooms apartment6614 Brissago,https://flatfox.ch/en/flat/6614-brissago/1848132/,CHF 1’200 per month,3,NaN,1961.0,2011.0,"Brissago: Appartamento 3 locali\n\nA Brissago,..."
1,1794995631-2,https://flatfox.ch/en/search/?east=10.5&is_swa...,4 rooms apartment9450 Altstätten,9450 Altstätten,Rental\n1’460 CHF,4 rooms apartment9450 Altstätten,https://flatfox.ch/en/flat/neufeldstrasse-7-94...,CHF 1’460 per month,4,81 m²,1975.0,NaN,Erster Nettomietzins geschenkt! Herzlich Willk...
2,1794995636-3,https://flatfox.ch/en/search/?east=10.5&is_swa...,3 rooms apartment2610 Saint-Imier,2610 Saint-Imier,Rental\n1’050 CHF,3 rooms apartment2610 Saint-Imier,https://flatfox.ch/en/flat/rue-de-la-chapelle-...,CHF 1’050 per month,3,60 m²,1919.0,2025.0,Charmant appartement de 3 pièces au cœur de Sa...
3,1794995639-4,https://flatfox.ch/en/search/?east=10.5&is_swa...,3 ½ rooms apartment9450 Altstätten,9450 Altstätten,Rental\n1’395 CHF,3 ½ rooms apartment9450 Altstätten,https://flatfox.ch/en/flat/ringgasse-27-9450-a...,CHF 1’395 per month,3 ½,78 m²,1920.0,2009.0,MIETZINSFREIE ZEIT GENIESSEN - TOLLE 3.5 ZIMME...
4,1794995645-5,https://flatfox.ch/en/search/?east=10.5&is_swa...,3 rooms apartment1260 Nyon,1260 Nyon,Rental\n2’100 CHF,3 rooms apartment1260 Nyon,https://flatfox.ch/en/flat/ch-de-la-redoute-36...,CHF 2’100 per month,3,80 m²,1986.0,2010.0,A louer sans caution - Joli logement bien situ...


## Count number of rows and columns in the data frame

In [3]:
# Dimension (rows, columns)
print('Dimension:', df.shape)

# Number of rows
print('Number of rows:', df.shape[0])

# Number of columns
print('Number of columns:', df.shape[1])

Dimension: (400, 13)
Number of rows: 400
Number of columns: 13


## Get data types (raw-format from web scraping)

In [4]:
# Get data types (since pandas 3.0, text columns have the data type 'str';
# in older versions of pandas, they have the data type 'object')
df.dtypes

web_scraper_order            str
web_scraper_start_url        str
rooms_card                   str
address                      str
price_card                   str
detail_url                   str
detail_url-href              str
gross_rent                   str
rooms_detail                 str
living_area                  str
year_built               float64
year_renovated           float64
description                  str
dtype: object

## Extract and save relevant information from raw data using regular expressions (regex)

### Extract number of rooms

The column `rooms_detail` contains values like `"3"` or `"3 ½"`. The text `"3 ½"` cannot be used in calculations, so the half room is converted to `.5` (e.g. `"3 ½"` → `3.5`).

In [5]:
# Extract values from 'rooms_detail' strings
rooms = []
for i in df['rooms_detail'].fillna('').astype(str):
    d1 = re.findall(r'(\d+(?:[.,]\d+)?)\s*(½)?', i)
    try:
        d2 = float(d1[0][0].replace(',', '.')) + (0.5 if d1[0][1] else 0.0)
    except:
        d2 = None
    rooms.append(d2)

# Save as new variable in the pandas data frame
df['rooms'] = pd.Series(rooms, dtype="float64")

# Print first 5 values
print(df['rooms_detail'].head(5), '\n')
print(df['rooms'].head(5))

0      3
1      4
2      3
3    3 ½
4      3
Name: rooms_detail, dtype: str 

0    3.0
1    4.0
2    3.0
3    3.5
4    3.0
Name: rooms, dtype: float64


### Extract living area

In [6]:
# Extract values from 'living_area' strings, e.g. '81 m²'
area = []
for i in df['living_area'].fillna('').astype(str):
    d1 = re.findall(r'(\d+)\s*m²', re.sub(r"['’]", '', i))  # remove thousands separators ' and ’
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    area.append(d2)

# Save as new variable in the pandas data frame
df['area'] = pd.Series(area, dtype="Int64")

# Print first 5 values
print(df['living_area'].head(5), '\n')
print(df['area'].head(5))

0      NaN
1    81 m²
2    60 m²
3    78 m²
4    80 m²
Name: living_area, dtype: str 

0    <NA>
1      81
2      60
3      78
4      80
Name: area, dtype: Int64


### Extract rental price

The column `gross_rent` contains values like `"CHF 1’200 per month"`. The thousands separator is the typographic apostrophe `’`, which is a different character from `'`. Both are removed, otherwise `"1’200"` would be read as `1`.

In [7]:
# Extract values from 'gross_rent' strings
price = []
for i in df['gross_rent'].fillna('').astype(str):
    d1 = re.findall('[0-9]+', re.sub(r"[\s'’]", '', i))  # remove spaces and thousands separators ' and ’
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df['price'] = pd.Series(price, dtype="Int64")

# Print first 5 values
print(df['gross_rent'].head(5), '\n')
print(df['price'].head(5))

0    CHF 1’200 per month
1    CHF 1’460 per month
2    CHF 1’050 per month
3    CHF 1’395 per month
4    CHF 2’100 per month
Name: gross_rent, dtype: str 

0    1200
1    1460
2    1050
3    1395
4    2100
Name: price, dtype: Int64


### Flag rent period, furnished and temporary listings

Some rents are given per week instead of per month. These rows are **kept** and only flagged with `rent_per_week`. Furnished and temporary listings are flagged as well, because they can bias the rent comparison.

In [8]:
# Rent period ('month', 'week', or missing for 'By request')
df['rent_period'] = df['gross_rent'].str.extract(r'per\s+(\w+)', expand=False)
df['rent_per_week'] = df['rent_period'].eq('week')

# Furnished / temporary flags from 'price_card'
df['furnished'] = df['price_card'].str.contains('Furnished', na=False)
df['temporary'] = df['price_card'].str.contains('Temporary', na=False)

print(df['rent_period'].value_counts(dropna=False), '\n')
print('Furnished:', df['furnished'].sum(), '| Temporary:', df['temporary'].sum(), '\n')

# Show the listings with weekly rents
df.loc[df['rent_per_week'], ['address', 'gross_rent', 'price']]

rent_period
month    389
NaN       10
week       1
Name: count, dtype: int64 

Furnished: 25 | Temporary: 19 



,address,gross_rent,price
5,6565 S. Bernardino,CHF 1’400 per week,1400


### Get data types of all variables including the new ones

In [9]:
df.dtypes

web_scraper_order            str
web_scraper_start_url        str
rooms_card                   str
address                      str
price_card                   str
detail_url                   str
detail_url-href              str
gross_rent                   str
rooms_detail                 str
living_area                  str
year_built               float64
year_renovated           float64
description                  str
rooms                    float64
area                       Int64
price                      Int64
rent_period                  str
rent_per_week               bool
furnished                   bool
temporary                   bool
dtype: object

## Count and identify missing values (if any)

In [10]:
# Count missing values
print(pd.isna(df).sum())

# Identify rows with missing values
df[df.isna().any(axis=1)].head()

web_scraper_order          0
web_scraper_start_url      0
rooms_card                 0
address                    0
price_card                 0
detail_url                 0
detail_url-href            0
gross_rent                 1
rooms_detail               1
living_area               18
year_built                 1
year_renovated           165
description                8
rooms                      1
area                      18
price                     10
rent_period               10
rent_per_week              0
furnished                  0
temporary                  0
dtype: int64


,web_scraper_order,web_scraper_start_url,rooms_card,address,price_card,detail_url,detail_url-href,gross_rent,rooms_detail,living_area,year_built,year_renovated,description,rooms,area,price,rent_period,rent_per_week,furnished,temporary
0,1794995628-1,https://flatfox.ch/en/search/?east=10.5&is_swa...,3 rooms apartment6614 Brissago,6614 Brissago,Rental\n1’200 CHF,3 rooms apartment6614 Brissago,https://flatfox.ch/en/flat/6614-brissago/1848132/,CHF 1’200 per month,3,NaN,1961.0,2011.0,"Brissago: Appartamento 3 locali\n\nA Brissago,...",3.0,<NA>,1200,month,False,False,False
1,1794995631-2,https://flatfox.ch/en/search/?east=10.5&is_swa...,4 rooms apartment9450 Altstätten,9450 Altstätten,Rental\n1’460 CHF,4 rooms apartment9450 Altstätten,https://flatfox.ch/en/flat/neufeldstrasse-7-94...,CHF 1’460 per month,4,81 m²,1975.0,NaN,Erster Nettomietzins geschenkt! Herzlich Willk...,4.0,81,1460,month,False,False,False
6,1794995653-7,https://flatfox.ch/en/search/?east=10.5&is_swa...,2 ½ rooms apartment8750 Glarus,8750 Glarus,Rental\n1’250 CHF,2 ½ rooms apartment8750 Glarus,https://flatfox.ch/en/flat/kirchstr-postgasse-...,CHF 1’250 per month,2 ½,62 m²,1861.0,NaN,1 Gratis-Nettomietzins + CHF 500.00 Umzugs-Gut...,2.5,62,1250,month,False,False,False
7,1794995657-8,https://flatfox.ch/en/search/?east=10.5&is_swa...,4 rooms apartment9450 Altstätten,9450 Altstätten,Rental\n1’400 CHF,4 rooms apartment9450 Altstätten,https://flatfox.ch/en/flat/neufeldstrasse-7-94...,CHF 1’400 per month,4,81 m²,1975.0,NaN,Erster Nettomietzins geschenkt! Zögern Sie nic...,4.0,81,1400,month,False,False,False
8,1794995662-9,https://flatfox.ch/en/search/?east=10.5&is_swa...,6 ½ rooms duplex7550 Scuol,7550 Scuol,Rental\n3’300 CHF,6 ½ rooms duplex7550 Scuol,https://flatfox.ch/en/flat/craista-143-7550-sc...,CHF 3’300 per month,6 ½,186 m²,1990.0,NaN,Geräumige 6.5-Zimmer-Maisonette-Wohnung mit Be...,6.5,186,3300,month,False,False,False


## Count and identify duplicated values (if any)

- `detail_url-href` identifies a listing. Rows with the same URL were scraped twice and are removed.
- Same address, rooms and rent can also be **different** flats in the same building (e.g. two flats on different floors), so these are only shown, not removed.

In [11]:
# Exact duplicates: same listing URL scraped more than once
print('Duplicated listing URLs:', df.duplicated(subset=['detail_url-href']).sum())

# Possible duplicates: same address, rooms and rent
dup_cols = ['address', 'rooms_detail', 'gross_rent']
print('Same address, rooms and rent:', df.duplicated(subset=dup_cols).sum())

# Identify rows with possible duplicates (all occurrences)
df[df.duplicated(subset=dup_cols, keep=False)].sort_values('address')[dup_cols + ['detail_url-href']]

Duplicated listing URLs: 0
Same address, rooms and rent: 1


,address,rooms_detail,gross_rent,detail_url-href
22,2830 Courrendlin,3 ½,CHF 880 per month,https://flatfox.ch/en/flat/rue-du-23-juin-15-2...
23,2830 Courrendlin,3 ½,CHF 880 per month,https://flatfox.ch/en/flat/rue-du-23-juin-15-2...


In [12]:
# Remove exact duplicates (same listing URL)
df = df.drop_duplicates(subset=['detail_url-href']).reset_index(drop=True)
print('Dimension after removing duplicated URLs:', df.shape)

Dimension after removing duplicated URLs: (400, 20)


## Topic 1: additional preparation

The following steps are not part of the Week 8 example. They are needed for the research question of Topic 1 (green premium of energy-renovated older apartments in Switzerland).

### Clean construction year and renovation year

- `year_built`: values of `0` or below 1000 (e.g. `189`) are set to missing. Only years up to 1989 are kept.
- Apartments built in 1990 are **removed**: the scrape filter (`max_year_built=1990`) includes 1990, but Topic 1 covers buildings built **before** 1990. Listings with an unknown construction year are kept (`year_built_clean` is missing).
- `year_renovated`: values of `0`, years in the future (after 2026) and years not after the construction year are set to missing.

In [13]:
# Convert to numbers (non-numeric values become missing)
df['year_built'] = pd.to_numeric(df['year_built'], errors='coerce')
df['year_renovated'] = pd.to_numeric(df['year_renovated'], errors='coerce')

# Clean construction year
df['year_built_clean'] = df['year_built'].where(df['year_built'].between(1000, 1989))

# Remove apartments built in 1990 or later (Topic 1: buildings built before 1990)
built_1990 = df['year_built'] >= 1990
print('Apartments built in 1990 or later (removed):', built_1990.sum())
df = df.loc[~built_1990].reset_index(drop=True)
print('Dimension after removal:', df.shape, '\n')

# Clean renovation year
valid_reno = (df['year_renovated'].between(1900, 2026)
              & (df['year_renovated'] > df['year_built_clean'].fillna(0)))
df['year_renovated_clean'] = df['year_renovated'].where(valid_reno)

# Show the values that were set to missing
print('year_built set to missing:')
print(df.loc[df['year_built'].notna() & df['year_built_clean'].isna(), ['address', 'year_built']], '\n')
print('year_renovated set to missing:')
print(df.loc[df['year_renovated'].notna() & df['year_renovated_clean'].isna(),
             ['address', 'year_built', 'year_renovated']])

Apartments built in 1990 or later (removed): 9
Dimension after removal: (391, 21) 

year_built set to missing:
                   address  year_built
69         1741 Cottens FR         0.0
74         1741 Cottens FR         0.0
93               1636 Broc         0.0
102  1635 La Tour-de-Trême         0.0
123  1635 La Tour-de-Trême         0.0
169  1470 Estavayer-le-lac         0.0
170  1470 Estavayer-le-lac         0.0
171  1470 Estavayer-le-lac         0.0
270  1635 La Tour-de-Trême         0.0
271          1700 Fribourg         0.0
318     3963 Crans-Montana         0.0
322   3032 Wohlen bei Bern       189.0
333         8637 Laupen ZH         0.0
355         8162 Steinmaur         0.0 

year_renovated set to missing:
                   address  year_built  year_renovated
69         1741 Cottens FR         0.0             0.0
74         1741 Cottens FR         0.0             0.0
93               1636 Broc         0.0             0.0
102  1635 La Tour-de-Trême         0.0             

### Plausibility check of living area and rent per m²

- Living areas below 10 m² or above 500 m² are set to missing (e.g. `1097 m²` is very likely a typo).
- `rent_per_m2` = gross rent / living area. It is only computed for **monthly** rents; for weekly rents it stays empty (the rows are kept and flagged).
- Note: the data only contains **gross** rent (including Nebenkosten), not net rent.

In [14]:
# Plausible living area
df['area_clean'] = df['area'].where(df['area'].between(10, 500))
print('Area set to missing:')
print(df.loc[df['area'].notna() & df['area_clean'].isna(), ['address', 'living_area']], '\n')

# Gross rent per m² (monthly rents only)
monthly = df['rent_period'].eq('month')
df['rent_per_m2'] = (df['price'] / df['area_clean']).where(monthly).round(2)

print(df['rent_per_m2'].describe())

Area set to missing:
          address living_area
148  4450 Sissach     1097 m² 

count        362.0
mean     23.794227
std       8.610858
min          11.45
25%        18.9425
50%           22.0
75%         26.225
max          79.17
Name: rent_per_m2, dtype: Float64


### Extract postcode and locality

The postcode and locality are extracted from `address` (e.g. `'8400 Winterthur'`). They are needed later to merge municipality-level data (e.g. SFSO "Regionalporträts", which covers all Swiss municipalities) as control variables for location.

In [15]:
# Postcode and locality from 'address', e.g. '8400 Winterthur'
df['zip'] = pd.to_numeric(df['address'].str.extract(r'^(\d{4})', expand=False), errors='coerce').astype('Int64')
df['locality'] = df['address'].str.extract(r'^\d{4}\s+(.*)$', expand=False).str.strip()

print('Listings without postcode:', df['zip'].isna().sum())
print('Number of different localities:', df['locality'].nunique(), '\n')

# Most frequent localities
print(df['locality'].value_counts().head(10))

Listings without postcode: 0
Number of different localities: 246 

locality
Kreuzlingen          7
Yverdon-les-Bains    6
Chur                 6
Sion                 6
Buchs SG             6
Schaffhausen         6
Bern                 6
Rheineck             5
Winterthur           5
Lausanne             5
Name: count, dtype: int64


### Renovation groups (keyword search in the description)

Each listing is assigned to one group:

1. **energy renovation**: the description contains an energy keyword (e.g. Wärmepumpe, Minergie, new windows, insulation)
2. **general renovation**: a general renovation keyword (e.g. renoviert, neue Küche) **or** a valid renovation year
3. **not renovated**: none of the above

The descriptions are in German, French and Italian, so keywords in all three languages are used. Keywords are matched as lower-case substrings. This is a simple heuristic: listings may be assigned to the wrong group (see limitations of Topic 1). Review a sample of each group and adjust the keyword lists if needed.

In [16]:
energy_keywords = [
    # German
    'energetisch', 'minergie', 'wärmepumpe', 'fassade', 'gebäudehülle', 'dämmung', 'gedämmt',
    'neue fenster', 'fenster ersetzt', 'fenstersanierung', '3-fach', '3fach', 'dreifach',
    'photovoltaik', 'solaranlage', 'pellet', 'neue heizung', 'heizung ersetzt',
    # French
    'pompe à chaleur', 'isolation thermique', 'façade', 'triple vitrage', 'photovoltaïque',
    'panneaux solaires', 'haute performance énergétique',
    # Italian
    'pompa di calore', 'facciate isolate', 'cappotto', 'tripli vetri', 'triplo vetro', 'fotovoltaico',
]

general_keywords = [
    # German
    'renoviert', 'renovation', 'saniert', 'sanierung', 'erneuert', 'modernisiert', 'umgebaut',
    'neue küche', 'neues bad', 'neue bad',
    # French
    'rénové', 'rénovation', 'refait', 'cuisine neuve',
    # Italian
    'ristrutturat', 'rinnovat', 'risanament',
]

desc = df['description'].fillna('').str.lower()
df['keyword_energy'] = desc.apply(lambda t: any(k in t for k in energy_keywords))
df['keyword_general'] = desc.apply(lambda t: any(k in t for k in general_keywords))

df['renovation_group'] = np.select(
    [df['keyword_energy'],
     df['keyword_general'] | df['year_renovated_clean'].notna()],
    ['energy renovation', 'general renovation'],
    default='not renovated')

print(df['renovation_group'].value_counts(), '\n')

# Mean gross rent per m² by renovation group (first descriptive comparison, no controls)
df.groupby('renovation_group')['rent_per_m2'].agg(['count', 'mean', 'median']).round(2)

renovation_group
general renovation    240
not renovated         133
energy renovation      18
Name: count, dtype: int64

,count,mean,median
renovation_group,,,
energy renovation,17,24.01,22.97
general renovation,225,24.29,22.38
not renovated,120,22.83,20.54


In [17]:
# Review: listings in the energy renovation group and the keywords found
def found_keywords(text, keywords):
    return ', '.join(k for k in keywords if k in text)

review = df[df['renovation_group'] == 'energy renovation'][['address', 'year_renovated_clean']].copy()
review['energy_keywords_found'] = desc[review.index].apply(lambda t: found_keywords(t, energy_keywords))
review

,address,year_renovated_clean,energy_keywords_found
44,6863 Besazio,2026.0,"facciate isolate, tripli vetri, fotovoltaico"
49,7141 Luven,2023.0,photovoltaik
69,1741 Cottens FR,NaN,triple vitrage
74,1741 Cottens FR,NaN,triple vitrage
87,7122 Valendas,NaN,wärmepumpe
122,9400 Rorschach,NaN,photovoltaik
137,8775 Hätzingen,2020.0,"neue fenster, pellet"
140,6500 Bellinzona,NaN,fotovoltaico
153,2013 Colombier NE,2024.0,"minergie, photovoltaïque"
166,2013 Colombier NE,2024.0,photovoltaïque


### Minergie (boolean)

The column `minergie` shows whether a listing is Minergie-certified. It combines two sources:

1. the attribute `minergie` of the listing on flatfox, queried in `01_flatfox_web_api.ipynb` and saved in `Data/flatfox_api_attributes.csv` (run that notebook first)
2. the word "Minergie" in the description

Values: `True` = attribute or description says Minergie; `False` = listing found on flatfox, without Minergie; `<NA>` = listing no longer online and no Minergie in the description (unknown). The data type `boolean` allows missing values.

In [18]:
# Listing ID from the detail URL, e.g. '.../6614-brissago/1848132/' -> 1848132
df['pk'] = df['detail_url-href'].str.extract(r'/(\d+)/?$', expand=False).astype('Int64')

# Attributes from the flatfox Web API
df_api = pd.read_csv('./Data/flatfox_api_attributes.csv', sep=',', encoding='utf-8')
df_api['pk'] = df_api['pk'].astype('Int64')
df = df.merge(df_api[['pk', 'http_status', 'attributes', 'rent_net']], on='pk', how='left')

# Source 1: attribute 'minergie' on flatfox
minergie_attribute = df['attributes'].fillna('').str.split(',').apply(lambda x: 'minergie' in x)

# Source 2: 'minergie' in the description
minergie_description = df['description'].fillna('').str.lower().str.contains('minergie')

# Boolean column: unknown (NA) if the listing was not found on flatfox
found = df['http_status'].eq(200)
df['minergie'] = pd.Series(pd.NA, index=df.index, dtype='boolean')
df.loc[found, 'minergie'] = minergie_attribute[found]
df.loc[minergie_description, 'minergie'] = True

print('minergie:')
print(df['minergie'].value_counts(dropna=False), '\n')
print('Source: attribute', minergie_attribute.sum(), '| description', minergie_description.sum(),
      '| both', (minergie_attribute & minergie_description).sum(), '\n')

# Minergie by renovation group
pd.crosstab(df['renovation_group'], df['minergie'].astype('str'), margins=True)

minergie:


minergie
False    386
True       5
Name: count, dtype: Int64 

Source: attribute 3 | description 2 | both 0 



minergie,False,True,All
renovation_group,,,
energy renovation,16,2,18
general renovation,237,3,240
not renovated,133,0,133
All,386,5,391


### Save data to file

In [19]:
df.to_csv('./Data/flatfox_pre1991_prepared.csv',
          sep=",",
          encoding='utf-8',
          index=False)

print('Saved:', df.shape[0], 'listings')

Saved: 391 listings


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [20]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:44:53
Python Version: 3.11.8
-----------------------------------
